# Clase 11: Visión Computacional en Tiempo Real — Conteo de Tráfico con YOLOv8

**FCyT UNCA — Programación Avanzada**

En esta sesión damos el salto de la **Clasificación** (decir *qué* hay en una imagen) a la **Detección y Rastreo** (localizar y *seguir* objetos en movimiento). Implementaremos un sistema de monitoreo de tráfico capaz de contar autos, motos y camiones en tiempo real.

---

## 1. El Salto Tecnológico: De CNN a YOLO

### 1.1 ¿Qué es YOLO?

Las redes convolucionales (CNN) que vimos antes clasifican *una imagen entera* con una etiqueta. Para video necesitamos algo más potente: **detectar múltiples objetos con sus coordenadas exactas, en tiempo real**.

**YOLO (You Only Look Once)** resuelve esto dividiendo la imagen en una grilla y prediciendo, en **una sola pasada por la red**, todas las cajas delimitadoras (*bounding boxes*) y sus clases. Esto lo hace órdenes de magnitud más rápido que los enfoques anteriores (R-CNN, Fast R-CNN).

Usaremos **YOLOv8** de Ultralytics, la versión más madura y bien documentada al momento de escribir este material.

### 1.2 Ventajas de YOLOv8

| Característica | Detalle |
|---|---|
| **Velocidad** | Hasta 160 FPS en GPU (modelo Nano). Ideal para video en tiempo real. |
| **Precisión competitiva** | El modelo Large supera a muchos detectores de dos etapas. |
| **Flexibilidad de tamaño** | 5 variantes: Nano (n), Small (s), Medium (m), Large (l), XLarge (x). |
| **80 clases pre-entrenadas** | Incluye personas, vehículos, animales, objetos cotidianos (dataset COCO). |
| **API unificada** | Detección, segmentación, pose y clasificación con la misma interfaz. |
| **Integración con `supervision`** | Anotación, rastreo y zonas de conteo listos para usar. |

### 1.3 Limitaciones de YOLOv8 (importante para ingeniería real)

| Limitación | Impacto práctico |
|---|---|
| **Oclusión** | Si un camión tapa una moto, el sistema la pierde. El tracker puede re-identificarla al reaparecer, pero no siempre. |
| **Objetos pequeños o lejanos** | El modelo Nano tiene dificultades con objetos que ocupan menos de ~20×20 píxeles. |
| **Condiciones de luz extremas** | Noche sin iluminación, sol directo a la cámara o lluvia intensa reducen la precisión. |
| **Ángulos no vistos** | El modelo fue entrenado mayormente con vistas frontales y laterales. Vistas cenitales (desde un dron) requieren fine-tuning. |
| **Clases fijas** | Solo detecta las 80 clases de COCO. Una maquinaria agrícola específica de Paraguay no está incluida. |
| **No hay comprensión semántica** | YOLO no «entiende» el tráfico: no sabe si un auto va en contramano o si está estacionado. |
| **Dependencia de hardware** | El modelo Large requiere GPU para correr en tiempo real. El Nano puede correr en CPU, pero a ~10-15 FPS. |

> **Regla de oro:** Para un proyecto real de ingeniería de tráfico, YOLOv8 es un excelente punto de partida, pero siempre debe ser validado con datos reales del sitio y complementado con lógica de negocio específica.

### 1.4 Conceptos clave de ingeniería

- **Detección vs. Tracking:** La detección localiza el objeto en cada cuadro de forma independiente. El *Tracking* (rastreo) le asigna un **ID único** para reconocer que el auto en el cuadro #1 es el mismo que en el cuadro #30. Sin tracking, el mismo auto puede ser contado múltiples veces.
- **ByteTrack:** El algoritmo de rastreo que usaremos. Asocia detecciones entre cuadros basándose en la posición y el tamaño de las cajas. Es robusto ante oclusiones breves.
- **Líneas de Cruce (*Tripwires*):** Definiremos coordenadas espaciales para contar un objeto **solo cuando cruce un umbral específico**, evitando contar el mismo vehículo por estar parado en el campo de visión.
- **Clases COCO:** El modelo reconoce `car (2)`, `motorcycle (3)`, `bus (5)`, `truck (7)` entre sus 80 clases. Filtraremos solo estas para nuestro caso de uso.


---
## 2. Instalación y Configuración del Entorno

In [ ]:
# BLOQUE 1: INSTALACIÓN
# ultralytics: contiene YOLOv8 y ByteTrack
# supervision: herramientas de anotación, zonas y conteo
!pip install ultralytics supervision --quiet

import cv2
import torch
import numpy as np
from pathlib import Path
from collections import defaultdict
from ultralytics import YOLO
import supervision as sv

print(f"supervision version: {sv.__version__}")

# Verificación de hardware
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"✅ GPU disponible: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU no disponible. Procesando en CPU (velocidad reducida).")
print(f">>> Dispositivo de inferencia: {device.upper()}")

---
## 3. Selección del Modelo: La Tabla de Velocidad vs. Precisión

YOLOv8 ofrece cinco variantes. La elección depende del hardware disponible:

| Modelo | Archivo | mAP50-95 | FPS (GPU T4) | Uso recomendado |
|--------|---------|----------|------------|------------------|
| Nano | `yolov8n.pt` | 37.3 | ~160 | Raspberry Pi, cámaras IP de bajo costo |
| Small | `yolov8s.pt` | 44.9 | ~120 | Laptops con GPU de gama media |
| Medium | `yolov8m.pt` | 50.2 | ~80 | Workstations, nuestra terminal de clase |
| Large | `yolov8l.pt` | 52.9 | ~50 | Servidores de producción |
| XLarge | `yolov8x.pt` | 53.9 | ~30 | Máxima precisión, sin restricción de tiempo |

> **Para esta clase usamos `yolov8n.pt`** (Nano) para que funcione bien en CPU si no hay GPU. Si tienen una RTX disponible, pueden probar con `yolov8m.pt` para mejor detección de motos en ángulos complicados.

In [ ]:
# BLOQUE 2: CARGA DEL MODELO
# El archivo .pt se descarga automáticamente la primera vez (~6 MB para Nano)
MODEL_NAME = "yolov8n.pt"  # Cambiar a "yolov8m.pt" si tienen GPU

model = YOLO(MODEL_NAME)
print(f"✅ Modelo {MODEL_NAME} cargado correctamente.")

# Explorar las clases disponibles en el modelo
print("\n--- Clases de vehículos disponibles en COCO ---")
clases_vehiculos = {2: 'car (auto)', 3: 'motorcycle (moto)', 5: 'bus (colectivo)', 7: 'truck (camión)'}
for idx, nombre in clases_vehiculos.items():
    print(f"  ID {idx}: {nombre}")

print(f"\n(Total de clases en el modelo: {len(model.names)})")

---
## 4. Configuración del Escenario de Tráfico

Antes de procesar el video, debemos definir **dónde** contamos. Para ello necesitamos conocer la resolución del video.

**¿Cómo elegir la posición de la línea?**
- Abrí el video y fijate en qué altura (coordenada Y) los vehículos están completamente visibles pero aún en movimiento.
- Evitá colocarla donde haya intersecciones o donde los autos se detengan (semáforos), porque el tracker podría perder el ID.
- Una línea **horizontal** funciona para calles con cámara frontal. Para rotondas, puede convenir usar una zona poligonal (`sv.PolygonZone`).

In [ ]:
# BLOQUE 3: INSPECCIÓN DEL VIDEO Y CONFIGURACIÓN DE LA LÍNEA

# ──────────────────────────────────────────────
# INSTRUCCIÓN: Colocá la ruta a tu video aquí
VIDEO_PATH = "trafico_video.mp4"
# ──────────────────────────────────────────────

# Verificar que el archivo existe
if not Path(VIDEO_PATH).exists():
    print(f"⚠️  Archivo '{VIDEO_PATH}' no encontrado.")
    print("    Descargá un video de prueba con:")
    print("    !wget -O trafico_video.mp4 'https://ultralytics.com/assets/road.mp4'")
else:
    # Leer metadatos del video
    cap = cv2.VideoCapture(VIDEO_PATH)
    ancho  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    alto   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    
    print(f"📹 Video: {VIDEO_PATH}")
    print(f"   Resolución: {ancho}×{alto} px")
    print(f"   FPS: {fps:.1f}")
    print(f"   Duración: {frames/fps:.1f} s ({frames} cuadros)")
    
    # Definir la línea de conteo a la mitad del alto del video
    LINEA_Y = alto // 2
    print(f"\n>>> Línea de conteo posicionada en Y={LINEA_Y} (mitad del video)")
    print(f"    Podés ajustar LINEA_Y manualmente si es necesario.")

---
## 5. El Motor de Visión: Detección + Rastreo + Conteo

Este es el núcleo del sistema. El flujo por cada cuadro es:

```
Cuadro de video
      │
      ▼
  YOLOv8 (inferencia)
      │  → Lista de detecciones: [caja, clase, confianza]
      ▼
  Filtro de clases vehiculares
      │  → Solo autos, motos, colectivos, camiones
      ▼
  ByteTrack (rastreo)
      │  → Asigna tracker_id único a cada vehículo
      ▼
  LineZone (conteo)
      │  → Si el centroide cruzó la línea → suma al contador
      ▼
  Anotadores visuales
      │  → Dibuja cajas, etiquetas y línea sobre el cuadro
      ▼
  Salida (ventana / archivo)
```


In [ ]:
# BLOQUE 4: INICIALIZACIÓN DE COMPONENTES

# ── Parámetros de detección ──────────────────
CONFIANZA_MINIMA = 0.35   # Umbral de confianza. Subir si hay muchos falsos positivos.
IOU_THRESHOLD    = 0.45   # Superposición máxima entre cajas (Non-Max Suppression)
CLASES_INTERES   = [2, 3, 5, 7]  # car, motorcycle, bus, truck

# Nombres legibles para mostrar en pantalla
ETIQUETAS = {2: 'Auto', 3: 'Moto', 5: 'Colectivo', 7: 'Camion'}

# ── Línea de conteo ──────────────────────────
# Ajustá LINEA_Y según inspección del video (Bloque 3)
LINEA_Y   = 400   # ← Modificar según el video
LINEA_X0  = 0
LINEA_X1  = 1280  # ← Ancho del video

linea_inicio = sv.Point(LINEA_X0, LINEA_Y)
linea_fin    = sv.Point(LINEA_X1, LINEA_Y)
line_zone    = sv.LineZone(start=linea_inicio, end=linea_fin)

# ── Rastreador (ByteTrack) ───────────────────
# track_thresh: confianza mínima para iniciar un track
# track_buffer: cuadros que espera antes de eliminar un track perdido
tracker = sv.ByteTrack(track_thresh=0.35, track_buffer=30)

# ── Anotadores visuales ──────────────────────
anotador_caja  = sv.BoxAnnotator(thickness=2)
anotador_label = sv.LabelAnnotator(text_thickness=1, text_scale=0.6, text_padding=3)
anotador_linea = sv.LineZoneAnnotator(
    thickness=3,
    text_thickness=2,
    text_scale=0.8,
    color=sv.Color.RED
)

print("✅ Todos los componentes inicializados.")
print(f"   Línea de conteo: Y={LINEA_Y}, de X={LINEA_X0} a X={LINEA_X1}")
print(f"   Clases monitoreadas: {[ETIQUETAS[c] for c in CLASES_INTERES]}")

In [ ]:
# BLOQUE 5: FUNCIÓN PRINCIPAL DE PROCESAMIENTO

def procesar_cuadro(frame):
    """
    Aplica detección, rastreo y conteo sobre un único cuadro de video.
    Retorna el cuadro anotado.
    """
    # 1. Inferencia con YOLOv8
    resultado = model(
        frame,
        device=device,
        conf=CONFIANZA_MINIMA,
        iou=IOU_THRESHOLD,
        verbose=False  # Suprimir logs por cuadro
    )[0]

    # 2. Convertir a formato supervision
    detecciones = sv.Detections.from_ultralytics(resultado)

    # 3. Filtrar solo clases vehiculares
    detecciones = detecciones[np.isin(detecciones.class_id, CLASES_INTERES)]

    # 4. Rastreo con ByteTrack (asigna tracker_id)
    detecciones = tracker.update_with_detections(detecciones)

    # 5. Disparar línea de conteo
    line_zone.trigger(detections=detecciones)

    # 6. Construir etiquetas: "Auto #3 (87%)"
    etiquetas = [
        f"{ETIQUETAS.get(class_id, 'Veh')} #{tracker_id} ({conf:.0%})"
        for class_id, tracker_id, conf in zip(
            detecciones.class_id,
            detecciones.tracker_id if detecciones.tracker_id is not None
                else [0]*len(detecciones),
            detecciones.confidence
        )
    ]

    # 7. Anotar visualmente el cuadro
    cuadro_anotado = anotador_caja.annotate(
        scene=frame.copy(), detections=detecciones
    )
    cuadro_anotado = anotador_label.annotate(
        scene=cuadro_anotado, detections=detecciones, labels=etiquetas
    )
    anotador_linea.annotate(cuadro_anotado, line_counter=line_zone)

    return cuadro_anotado


def ejecutar_conteo(video_path, guardar_salida=False):
    """
    Procesa el video completo cuadro por cuadro.
    
    Args:
        video_path (str): Ruta al archivo de video.
        guardar_salida (bool): Si True, guarda el video anotado en disco.
    """
    path = Path(video_path)
    if not path.exists():
        print(f"❌ Archivo no encontrado: {video_path}")
        return

    # Resetear contadores del tracker y la línea
    tracker.reset()
    line_zone.in_count  = 0
    line_zone.out_count = 0

    generador = sv.get_video_frames_generator(source_path=str(path))
    info_video = sv.VideoInfo.from_video_path(str(path))

    # Configurar escritura de video de salida (opcional)
    escritor = None
    if guardar_salida:
        ruta_salida = path.stem + "_anotado.mp4"
        escritor = sv.VideoSink(
            target_path=ruta_salida,
            video_info=info_video
        )
        print(f"📼 Guardando salida en: {ruta_salida}")

    print("▶️  Iniciando procesamiento. Presioná 'q' para detener.")
    print("-" * 50)

    try:
        for num_cuadro, cuadro in enumerate(generador):
            cuadro_anotado = procesar_cuadro(cuadro)

            if guardar_salida and escritor:
                escritor.write_frame(cuadro_anotado)

            cv2.imshow("Terminal de Tráfico — FCyT UNCA", cuadro_anotado)

            # Mostrar progreso cada 30 cuadros
            if num_cuadro % 30 == 0:
                print(
                    f"Cuadro {num_cuadro:5d} | "
                    f"Entrando: {line_zone.in_count:3d} | "
                    f"Saliendo: {line_zone.out_count:3d}"
                )

            if cv2.waitKey(1) & 0xFF == ord('q'):
                print("\n⏹️  Procesamiento detenido por el usuario.")
                break

    finally:
        cv2.destroyAllWindows()
        if escritor:
            escritor.__exit__(None, None, None)

    print("\n" + "=" * 50)
    print("          REPORTE FINAL DE TRÁFICO")
    print("=" * 50)
    print(f"  Vehículos que entraron (↓): {line_zone.in_count}")
    print(f"  Vehículos que salieron (↑): {line_zone.out_count}")
    print(f"  Total contabilizado:        {line_zone.in_count + line_zone.out_count}")
    print("=" * 50)


print("✅ Funciones definidas. Ejecutar la celda siguiente para procesar el video.")

In [ ]:
# BLOQUE 6: EJECUCIÓN
# Descomentar la línea correspondiente:

# Opción A: Solo visualización en ventana
# ejecutar_conteo('trafico_video.mp4')

# Opción B: Visualización + guardar video anotado en disco
# ejecutar_conteo('trafico_video.mp4', guardar_salida=True)

# Opción C: Descargar video de prueba de Ultralytics y procesarlo
# !wget -q -O trafico_video.mp4 'https://ultralytics.com/assets/road.mp4'
# ejecutar_conteo('trafico_video.mp4')

print("↑ Descomentá una de las opciones y re-ejecutá esta celda.")

---
## 6. Análisis: ¿A cuántos FPS corre el sistema?

Para saber si nuestro sistema es útil en tiempo real, necesitamos medir su velocidad. Una moto a 60 km/h recorre **16.7 metros por segundo**. Si la zona de visión de la cámara abarca 10 metros, la moto estará en cuadro apenas **0.6 segundos**. A 10 FPS, eso es solo 6 cuadros para detectar, rastrear y contar.

In [ ]:
# BLOQUE 7: BENCHMARK DE VELOCIDAD
import time

def medir_fps(video_path, n_cuadros=100):
    """
    Procesa los primeros n_cuadros del video y calcula el FPS promedio.
    """
    path = Path(video_path)
    if not path.exists():
        print(f"❌ Archivo no encontrado: {video_path}")
        return

    tracker.reset()
    generador = sv.get_video_frames_generator(source_path=str(path))

    tiempos = []
    for i, cuadro in enumerate(generador):
        if i >= n_cuadros:
            break
        t0 = time.perf_counter()
        procesar_cuadro(cuadro)
        tiempos.append(time.perf_counter() - t0)

    if not tiempos:
        print("No se pudo procesar ningún cuadro.")
        return

    fps_prom = 1 / np.mean(tiempos)
    fps_min  = 1 / np.max(tiempos)
    fps_max  = 1 / np.min(tiempos)

    print(f"\n{'─'*40}")
    print(f"  BENCHMARK — {n_cuadros} cuadros ({MODEL_NAME})")
    print(f"{'─'*40}")
    print(f"  FPS promedio : {fps_prom:.1f}")
    print(f"  FPS mínimo   : {fps_min:.1f}")
    print(f"  FPS máximo   : {fps_max:.1f}")
    print(f"  Dispositivo  : {device.upper()}")

    # Evaluación para tráfico urbano
    if fps_prom >= 25:
        print(f"  ✅ Suficiente para tiempo real (≥25 FPS)")
    elif fps_prom >= 10:
        print(f"  ⚠️  Marginal. Puede perder motos a alta velocidad (<25 FPS)")
    else:
        print(f"  ❌ Insuficiente para tiempo real. Considerar modelo Nano o GPU.")
    print(f"{'─'*40}")


# Descomentar para ejecutar el benchmark:
# medir_fps('trafico_video.mp4', n_cuadros=100)

---
## 7. Preguntas de Análisis

Respondan individualmente o en grupo y entreguen junto con el código de la tarea:

1. **FPS vs. velocidad vehicular:** Con los FPS medidos en el Bloque 7, ¿cuántos cuadros tienen para detectar una moto que viaja a 80 km/h si la cámara cubre 8 metros de calle?

2. **El problema de la oclusión:** Describí con tus palabras qué pasa cuando un camión pasa por delante de una moto en el mismo cuadro. ¿Cómo afecta esto al `in_count`? ¿Qué parámetro de ByteTrack ayudaría a mitigarlo?

3. **Limitación de clases:** El modelo COCO no incluye mototaxis, carretillas tiradas por caballos ni tractores. ¿Qué estrategia usarías para que el sistema también los detecte? (Pista: investigá el concepto de *fine-tuning*)

4. **Aplicación local:** ¿Cómo usarían los datos de `in_count` y `out_count` por hora del día para optimizar los tiempos de semáforo en una intersección de tu ciudad?

---
## 8. Tarea: Interfaz de Conteo por Tipo de Vehículo

### Objetivo
Extender el sistema para contar **por separado** autos, motos y camiones/colectivos, y mostrar esa información en una interfaz visual sobre el video.

### Requisitos

**Requisito 1 — Conteo diferenciado:**
En lugar de un único `in_count`, el sistema debe mantener contadores separados:
```python
conteo = {'Auto': 0, 'Moto': 0, 'Camion/Colectivo': 0}
```
Cuando un vehículo cruce la línea, sumar al contador de su clase.

**Requisito 2 — Panel de información en el video:**
Superponer un recuadro semitransparente en la esquina superior derecha del video con los conteos actualizados en tiempo real. Debe verse similar a:
```
┌─────────────────┐
│  CONTEO ACTUAL  │
│  Autos:    12   │
│  Motos:     7   │
│  Cam/Col:   3   │
└─────────────────┘
```
Podés usar `cv2.rectangle` y `cv2.putText` para dibujarlo.

**Requisito 3 — Colores por clase:**
Cada tipo de vehículo debe tener un color diferente en su bounding box:
- Auto: azul
- Moto: verde
- Camión/Colectivo: rojo

### Pistas técnicas

```python
# Para dibujar un rectángulo semitransparente:
overlay = frame.copy()
cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 0, 0), -1)  # relleno negro
frame = cv2.addWeighted(overlay, 0.4, frame, 0.6, 0)        # mezcla con transparencia

# Para obtener los IDs de vehículos que cruzaron la línea:
# line_zone.trigger() devuelve dos arrays booleanos:
cruzo_entrando, cruzo_saliendo = line_zone.trigger(detections=detecciones)
# cruzo_entrando[i] == True si la detección i cruzó en dirección de entrada

# Para asignar colores por clase con supervision:
paleta = sv.ColorPalette.from_hex(['#3498DB', '#2ECC71', '#E74C3C'])
# Mapear class_id a índice de color:
mapa_color = {2: 0, 3: 1, 5: 2, 7: 2}  # auto→azul, moto→verde, bus/truck→rojo
```

### Entregable
- Un notebook `.ipynb` con el código completo y comentado.
- Un breve informe (puede ser en celdas Markdown dentro del mismo notebook) respondiendo las preguntas del Apartado 7.
- **Opcional (+puntos):** Guardar el video anotado con `guardar_salida=True` y entregarlo junto al notebook.

In [ ]:
# BLOQUE 8: ESPACIO DE TRABAJO — TAREA
# Comenzá tu implementación aquí.
# Te dejamos la estructura base como punto de partida:

# ── Contadores por clase ─────────────────────
conteo_entrada = defaultdict(int)  # {'Auto': 0, 'Moto': 0, ...}
conteo_salida  = defaultdict(int)

# ── Paleta de colores por clase ──────────────
# TODO: Definir colores para cada tipo de vehículo

def dibujar_panel(frame, conteo_e, conteo_s):
    """
    TODO: Superponer el panel de conteo sobre el frame.
    Retornar el frame modificado.
    """
    pass


def procesar_cuadro_v2(frame):
    """
    TODO: Versión extendida de procesar_cuadro que:
    1. Detecta y rastrea vehículos.
    2. Diferencia conteos por tipo de vehículo.
    3. Usa colores distintos por clase.
    4. Superpone el panel de información.
    """
    pass


print("💡 Implementá las funciones `dibujar_panel` y `procesar_cuadro_v2`.")
print("   Luego copiá la lógica de `ejecutar_conteo` y adaptala para usar `procesar_cuadro_v2`.")